# Task: Prepare the merged data for the "PAP" analysis

Input: `data/intermediate/` (from `merge.ipynb`). Output: `data/clean/`.

* treatment indicators `CV`, `SkillRecs`, `DemandRecs`
* strata = municipality × above/below-median age × program (PAP 3.2, 5.1)
* pool of socio-demographic candidate controls for double-selection LASSO (PAP 5.1)
* the five outcome families (PAP 4.3 / 5.3) and their survey/admin measures
* heterogeneity variables: gender and age (PAP 5.2)

No outlier trimming, winsorizing, imputation or sample restrictions beyond what the PAP states. 

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

pd.set_option("display.width", 200, "display.max_columns", 60)

# repo root = first parent folder that contains data/raw (works from code/cleaning/ or anywhere below the root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
INTERMEDIATE = ROOT / "data" / "intermediate"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents = True, exist_ok=True)

# First SkillLab sign-up in the data is 2025-04-30; treat employment from this month on as "post".
POST_START_MONTH = 202505

## 1. Load merged data

In [2]:
df = pd.read_csv(INTERMEDIATE / "merged_person.csv", low_memory = False,
                parse_dates = ["fecha_nacimiento", "sl_signup_date"])
emp = pd.read_csv(INTERMEDIATE / "emp_panel_long.csv", low_memory = False, parse_dates = ["date_m"])

# CSV turns empty strings into NaN; the registration data use "" as a real "not answered" category
str_cols = ["pueblo_originario", "etnia", "nivel_educativo", "trabaja_actualmente",
            "experiencia_laboral_anterior", "discapacidad", "hijos_menores", "región"]
df[str_cols] = df[str_cols].fillna("")
print(df.shape, "| employment panel:", emp.shape)
assert df["id_SSE"].is_unique

(1034347, 114) | employment panel: (136344, 12)


## 2. Treatment indicators
Arms are mutually exclusive: `treat` 0 = Control, 1 = T1 (CV), 2 = T2 (CV + Recs), 3 = T3 (CV + Recs + In-demand).  

The specification is `Y = b0 + b1·CV + b2·SkillRecs + b3·DemandRecs + strata FE + X`.  

`CV = 1[treat==1]`, `SkillRecs = 1[treat==2]`, `DemandRecs = 1[treat==3]`

In [3]:
assert df["treat"].isin([0, 1, 2, 3]).all()
df["CV"]         = (df["treat"] == 1).astype(int)
df["SkillRecs"]  = (df["treat"] == 2).astype(int)
df["DemandRecs"] = (df["treat"] == 3).astype(int)
df["arm"] = df["treat"].map({0: "Control", 1: "T1_CV", 2: "T2_SkillRecs", 3: "T3_DemandRecs"})
df[["CV", "SkillRecs", "DemandRecs"]].sum()

CV            258587
SkillRecs     258587
DemandRecs    258587
dtype: int64

## 3. Strata (PAP 3.2 / 5.1): municipality × age (above/below median) × program
`edad_above_med` (age above median) and `fomentar` (1 = Fomentar, 0 = VAT) were constructed by the research team in the randomization file;
we use them as-is. Municipality is `muni_id`.

In [4]:
assert set(df["fomentar"].unique()) <= {0.0, 1.0}
assert set(df["edad_above_med"].unique()) <= {0.0, 1.0}
assert (df["fomentar"] == (df["grupo"] == "fomentar")).all()

print("missing muni_id:", df["muni_id"].isna().sum(), f"({df['muni_id'].isna().mean():.1%} of randomized sample)")

df["strata"] = (df["muni_id"].astype("Int64").astype(str) + "_"
                + df["edad_above_med"].astype(int).astype(str) + "_"
                + df["fomentar"].astype(int).astype(str))
df.loc[df["muni_id"].isna(), "strata"] = np.nan          # strata undefined without a municipality; NOT imputed
print("number of strata:", df["strata"].nunique())
df.groupby("arm")["strata"].apply(lambda s: s.notna().mean()).round(4)

missing muni_id: 158683 (15.3% of randomized sample)
number of strata: 2290


arm
Control          0.8466
T1_CV            0.8466
T2_SkillRecs     0.8466
T3_DemandRecs    0.8466
Name: strata, dtype: float64

## 4. Heterogeneity variables (PAP 5.2)
Gender (`mujer`: from `sexo`, already built) and age (`edad`, `edad_above_med`). The mobility analysis needs home-to-course and home-to-job
distances; no course-location or firm-location coordinates are in `data/raw/`.

In [5]:
assert (df["mujer"] == (df["sexo"] == "F").astype(float)).all()
df["female"] = df["mujer"].astype(int)
print(df["sexo"].value_counts())
print("\nage summary:"); print(df["edad"].describe())
print("\nage > 100 (data error to report, left untouched):", (df["edad"] > 100).sum())

sexo
F    649163
M    385058
X       126
Name: count, dtype: int64

age summary:
count    1.034347e+06
mean     3.229078e+01
std      9.403853e+00
min      1.800000e+01
25%      2.500000e+01
50%      3.000000e+01
75%      3.800000e+01
max      2.025000e+03
Name: edad, dtype: float64

age > 100 (data error to report, left untouched): 1


## 5. Pool of candidate socio-demographic controls (PAP 5.1)
The PAP says controls `X` are selected by **double-selection LASSO**. Here we only build the candidate pool from the pre-treatment
characteristics in the Ministry registration data: numeric variables plus dummies for each category (blank string = its own
"missing" category, so no observation is dropped). `strata` fixed effects are *not* part of this pool.

In [6]:
numeric_candidates = ["edad", "female", "has_children", "educ_primary", "educ_secondary", "work_experience"]
categorical_candidates = ["nivel_educativo", "trabaja_actualmente", "experiencia_laboral_anterior",
                          "discapacidad", "hijos_menores", "región"]

# indigenous / ethnic self-identification: only an indicator of any answer (36 + 5 sparse categories)
df["any_pueblo_originario"] = (df["pueblo_originario"].str.strip() != "").astype(int)
df["any_etnia"]             = (df["etnia"].str.strip() != "").astype(int)
numeric_candidates += ["any_pueblo_originario", "any_etnia"]

dummies = pd.get_dummies(df[categorical_candidates].replace("", "missing").astype(str),
                         prefix=categorical_candidates, dtype=int)
dummies.columns = [c.replace(" ", "_") for c in dummies.columns]

df = pd.concat([df, dummies], axis=1)
X_CANDIDATES = numeric_candidates + list(dummies.columns)

print(len(X_CANDIDATES), "candidate controls")
print("missing values in numeric candidates:"); print(df[numeric_candidates].isna().sum()[lambda s: s > 0])

31 candidate controls
missing values in numeric candidates:
Series([], dtype: int64)


## 6. Survey outcomes (PAP 4.3, 5.3) — Opinaia phone survey
Mapping of survey questions to the PAP outcome families (question wording: `data/raw/RCT/opinaia/` questionnaire).

| PAP family | Variable | Question | Type |
|---|---|---|---|
| 1. Skills knowledge & career aspirations | `new_skill` | P4 noticed a new skill | binary |
| | `new_career_path` | P6 considered a new career path | binary |
| 2. Job-search behavior | `portal_visits` | P1 portal visits, last 2 weeks | count |
| | `search_hours` | P2 hours searching, last week | continuous |
| | `n_applications` | P3 applications, last month | count |
| 3. Reservation wage | `reservation_wage` | P8 minimum monthly wage (30 min from home) | continuous |
| 4. Employment | `days_worked` | P0 days worked for pay, last month | count |
| | `job_offer` | P10 received a job offer, last month | binary |
| | `job_prob_3m` | P9 prob. of finding a job in 3 months (1–10) | scale |
| 5. Mental health | — | not in the survey instrument | **missing, see TODO** |

"Don't know" answers are already blank in the numeric columns (the `*_dk` flags mark them); they stay `NaN`.

In [7]:
sv = df["in_survey"] == 1

def yes_no(s):
    return s.map({"Sí": 1.0, "Si": 1.0, "No": 0.0})

def scale_1_10(s):
    # '1 - Imposible', '2', ..., '10 - Muy posible'
    return pd.to_numeric(s.astype("string").str.extract(r"^(\d+)")[0], errors="coerce")

df["new_skill"]        = yes_no(df["sv_P4"])
df["new_career_path"]  = yes_no(df["sv_P6"])
df["portal_visits"]    = df["sv_P1"]
df["search_hours"]     = df["sv_P2"]
df["n_applications"]   = df["sv_P3"]
df["reservation_wage"] = df["sv_P8"]
df["days_worked"]      = df["sv_P0"]
df["job_offer"]        = yes_no(df["sv_P10"])
df["job_prob_3m"]      = scale_1_10(df["sv_P9"])

SURVEY_OUTCOMES = {
    "skills_aspirations": ["new_skill", "new_career_path"],
    "job_search":         ["portal_visits", "search_hours", "n_applications"],
    "reservation_wage":   ["reservation_wage"],
    "employment":         ["days_worked", "job_offer", "job_prob_3m"],
}

all_survey = [v for vs in SURVEY_OUTCOMES.values() for v in vs]
display(df.loc[sv, all_survey].describe().T.round(2))
print("Unparsed answers (non-missing raw but NaN after recode):")

for raw, new in [("sv_P4", "new_skill"), ("sv_P6", "new_career_path"), ("sv_P10", "job_offer"), ("sv_P9", "job_prob_3m")]:
    print(f"  {new:16s}", int((df.loc[sv, raw].notna() & df.loc[sv, new].isna()).sum()))

,count,mean,std,min,25%,50%,75%,max
new_skill,1021.0,0.428991,0.495175,0.0,0.0,0.0,1.0,1.0
new_career_path,1021.0,0.631734,0.482571,0.0,0.0,1.0,1.0,1.0
portal_visits,967.0,1.681489,3.60525,0.0,0.0,0.0,2.0,60.0
search_hours,967.0,5.754912,11.127574,0.0,0.0,2.0,6.0,124.0
n_applications,985.0,5.922843,13.829101,0.0,0.0,2.0,6.0,200.0
reservation_wage,842.0,646331.351544,443116.451087,24000.0,400000.0,600000.0,800000.0,4999999.0
days_worked,1016.0,11.357283,10.868371,0.0,0.0,10.0,20.0,31.0
job_offer,1021.0,0.227228,0.419247,0.0,0.0,0.0,0.0,1.0
job_prob_3m,1021.0,5.951028,2.550192,1.0,5.0,6.0,8.0,10.0


Unparsed answers (non-missing raw but NaN after recode):
  new_skill        0
  new_career_path  0
  job_offer        0
  job_prob_3m      0


### 6b. Standardized outcomes (PAP 5.3: effects in SD of the outcome)
The PAP reports effects in standard deviations ("continuous case ... 0.075–0.225 SD"). Each non-binary survey outcome is standardized with the
**control-group** mean and SD among survey respondents. Binary outcomes stay in levels (percentage points).

In [8]:
continuous = ["portal_visits", "search_hours", "n_applications", "reservation_wage", "days_worked", "job_prob_3m"]
ctrl = sv & (df["treat"] == 0)

for v in continuous:
    mu, sd = df.loc[ctrl, v].mean(), df.loc[ctrl, v].std()
    df[f"{v}_z"] = (df[v] - mu) / sd
    print(f"{v:18s} control mean = {mu:12.2f},  \
          sd = {sd:12.2f},  \
            (n control = {df.loc[ctrl, v].notna().sum()})")

portal_visits      control mean =         1.34,            sd =         2.42,              (n control = 93)
search_hours       control mean =         5.80,            sd =        10.59,              (n control = 94)
n_applications     control mean =         5.86,            sd =        20.72,              (n control = 97)
reservation_wage   control mean =    601095.89,            sd =    288923.10,              (n control = 73)
days_worked        control mean =         9.18,            sd =        10.61,              (n control = 95)
job_prob_3m        control mean =         5.81,            sd =         2.58,              (n control = 100)


/var/folders/nc/3ytrhrfn76s1z1kxjy6wfkzh0000gq/T/ipykernel_17692/2565870095.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{v}_z"] = (df[v] - mu) / sd
/var/folders/nc/3ytrhrfn76s1z1kxjy6wfkzh0000gq/T/ipykernel_17692/2565870095.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{v}_z"] = (df[v] - mu) / sd


## 7. Administrative outcomes (PAP 4.2) — Ministry of Human Capital
* Courses: `enrolled_in_course` (file 1); CV updates on Portal Empleo: `updated_cv` (file 4); also files 2 and 3 (portal job application, contact permission).
* Formal employment: from the person-month panel (`active_employment`, 2024-01 … 2025-07).

**Coding of absent ids.** Files 1–4 list only ids with a `1`; an id missing from a file is coded `0` (the Ministry file is a list of
people with the event). The employment panel, however, covers only 6,026 ids — the same ~0.6% of every arm — so an id *outside* the panel is **unknown, not unemployed**
and is left `NaN`. Both rules are listed as open questions in `TODO.md`.

In [9]:
admin_flags = ["enrolled_in_course", "updated_cv", "applied_to_portal_job", "allowed_companies_contact"]
for v in admin_flags:
    df[v] = df[v].fillna(0).astype(int)
    
display(df.groupby("arm")[admin_flags].mean().round(4))

,enrolled_in_course,updated_cv,applied_to_portal_job,allowed_companies_contact
arm,,,,
Control,0.0092,0.0318,0.0023,0.0082
T1_CV,0.0094,0.0323,0.0023,0.0086
T2_SkillRecs,0.0095,0.0317,0.0023,0.0084
T3_DemandRecs,0.0094,0.0322,0.0021,0.0085


In [10]:
# Person-level employment measures from the panel
emp = emp.copy()
emp["post"] = emp["month"] >= POST_START_MONTH

# One row per person-month: a person with several employers is employed if ANY employer is active
pm = emp.groupby(["id_SSE", "month", "post"], as_index=False)["active_employment"].max()
print("person-months:", len(pm), "| duplicated person-month after collapsing employers:", pm.duplicated(["id_SSE", "month"]).sum())

emp_person = (pm.groupby("id_SSE")
                .apply(lambda g: pd.Series({
                    "emp_months_observed_post": int(g["post"].sum()),
                    "emp_any_post": float(g.loc[g["post"], "active_employment"].max()) if g["post"].any() else np.nan,
                    "emp_share_months_post": g.loc[g["post"], "active_employment"].mean() if g["post"].any() else np.nan,
                    "emp_any_pre":  float(g.loc[~g["post"], "active_employment"].max()) if (~g["post"]).any() else np.nan,
                }), include_groups=False)
                .reset_index())

df = df.merge(emp_person, on="id_SSE", how="left", validate="1:1")
print("persons with panel data:", df["emp_any_post"].notna().sum())
df.groupby("arm")[["emp_any_pre", "emp_any_post", "emp_share_months_post"]].mean().round(4)

person-months: 114494 | duplicated person-month after collapsing employers: 0
persons with panel data: 6026


,emp_any_pre,emp_any_post,emp_share_months_post
arm,,,
Control,0.8730,0.5537,0.4574
T1_CV,0.8723,0.5475,0.4641
T2_SkillRecs,0.8766,0.5837,0.4933
T3_DemandRecs,0.8597,0.5798,0.4849


## 8. SkillLab take-up (PAP 4.1)
Take-up = signed up on SkillLab (`sl_signup`). Only treated arms can sign up; Control has no access in 2025, so Control is `0`.
Because only 1,279 of 3,363 sign-ups can be linked to an `id_SSE` through the crosswalks, this is a **lower bound**.

In [11]:
usage_cols = [c for c in df.columns if c.startswith("sl_n_") or c == "sl_signup"]
df[usage_cols] = df[usage_cols].fillna(0)

df["took_up"] = df["sl_signup"].astype(int)
df["cv_exported"] = (df["sl_n_cv_exports"] > 0).astype(int)
display(df.groupby("arm")[["took_up", "cv_exported", "sl_n_sessions", "sl_n_unique_skills"]].mean().round(4))

/var/folders/nc/3ytrhrfn76s1z1kxjy6wfkzh0000gq/T/ipykernel_17692/1393191726.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["took_up"] = df["sl_signup"].astype(int)
/var/folders/nc/3ytrhrfn76s1z1kxjy6wfkzh0000gq/T/ipykernel_17692/1393191726.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["cv_exported"] = (df["sl_n_cv_exports"] > 0).astype(int)


,took_up,cv_exported,sl_n_sessions,sl_n_unique_skills
arm,,,,
Control,0.0000,0.0000,0.0000,0.0000
T1_CV,0.0016,0.0004,0.0017,0.0262
T2_SkillRecs,0.0017,0.0004,0.0018,0.0400
T3_DemandRecs,0.0016,0.0004,0.0017,0.0268


## 9. Final checks

In [12]:
assert df["id_SSE"].is_unique and len(df) == 1_034_347
assert df.groupby("id_SSE")[["CV", "SkillRecs", "DemandRecs"]].sum().sum(axis=1).max() <= 1
assert (df.loc[df["treat"] == 0, "took_up"] == 0).all(), "control users with SkillLab sign-up"
# arms balanced on pre-treatment covariates in the full randomized sample (sanity check, not a PAP table)
display(df.groupby("arm")[["edad", "female", "fomentar", "has_children", "educ_secondary", "work_experience"]].mean().round(3))
print("survey sample by arm:"); display(df.loc[sv].groupby("arm").size())

,edad,female,fomentar,has_children,educ_secondary,work_experience
arm,,,,,,
Control,32.286,0.626,0.633,0.464,0.445,0.406
T1_CV,32.271,0.627,0.633,0.464,0.449,0.405
T2_SkillRecs,32.309,0.628,0.633,0.465,0.448,0.405
T3_DemandRecs,32.297,0.629,0.633,0.467,0.447,0.406


survey sample by arm:


arm
Control          100
T1_CV            320
T2_SkillRecs     301
T3_DemandRecs    300
dtype: int64

## 10. Save to `data/clean/`
* `analysis_full.csv`:    all randomized beneficiaries (admin outcomes, take-up).
* `analysis_survey.csv`:  phone-survey respondents (survey outcomes).
* `analysis_vars.json`":  variable lists (treatments, strata, candidate controls, outcome families) for the analysis notebooks.

In [13]:
drop_raw = [c for c in df.columns if c.startswith("sv_") and c not in ("sv_Grupo",)]   # raw survey columns recoded above
keep = [c for c in df.columns if c not in drop_raw]
df[keep].to_csv(CLEAN / "analysis_full.csv", index=False)
df.loc[sv, keep].to_csv(CLEAN / "analysis_survey.csv", index=False)

variables = {
    "treatments": ["CV", "SkillRecs", "DemandRecs"],
    "strata": "strata",
    "x_candidates": X_CANDIDATES,
    "survey_outcomes": SURVEY_OUTCOMES,
    "admin_outcomes": {"courses": ["enrolled_in_course"], "cv_update": ["updated_cv"],
                       "formal_employment": ["emp_any_post", "emp_share_months_post"]},
    "heterogeneity": ["female", "edad_above_med"],
    "post_start_month": POST_START_MONTH,
}
(CLEAN / "analysis_vars.json").write_text(json.dumps(variables, indent=2))
sorted(p.name for p in CLEAN.iterdir())

['analysis_full.csv', 'analysis_survey.csv', 'analysis_vars.json']